# 调整和确定可视化

In [2]:
import os
import re
import pickle
import json
import random
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import colorsys
from IPython.lib.display import IFrame
from causalnex.structure import StructureModel
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE

## 导入原图

In [3]:
net = nx.drawing.nx_pydot.read_dot('supporting_files/graph.dot')
g = StructureModel(net)

In [4]:
print(g.edges)

[('area0', 'area1'), ('area0', 'area2'), ('area0', 'area3'), ('area1', 'area2'), ('area1', 'area3'), ('area2', 'area3'), ('cluster0', 'area0'), ('cluster0', 'cluster2'), ('cluster0', 'cluster3'), ('cluster0', 'income'), ('cluster0', 'state'), ('cluster1', 'area1'), ('cluster1', 'cluster2'), ('cluster1', 'cluster3'), ('cluster1', 'income'), ('cluster2', 'cluster3'), ('home_road_density_0', 'home_landusemix_0'), ('home_road_density_0', 'home_openratio_wide_0'), ('home_road_density_0', 'home_road_density_1'), ('home_road_density_0', 'income'), ('home_openratio_wide_0', 'home_landusemix_0'), ('home_bus_0', 'home_landusemix_0'), ('work_road_density_0', 'home_road_density_0'), ('work_road_density_0', 'work_landusemix_0'), ('work_subway_0', 'avg_daily_tube_freq1'), ('work_subway_0', 'avg_daily_tube_freq2'), ('work_bus_0', 'work_landusemix_0'), ('work_bus_0', 'work_subway_0'), ('commute_distance_0', 'area3'), ('commute_distance_0', 'avg_daily_slow_freq0'), ('commute_distance_0', 'avg_daily_slo

In [3]:
viz = plot_structure(g)
viz.show('supporting_files/03_plot.html')

supporting_files/03_plot.html


## 可视化调整

In [4]:
# 获取每个节点名称
nodes = viz.nodes
node_positions = []

# 提取每个节点的x,y坐标
for node in nodes:
    node_positions.append(node['id'])

# 打印节点坐标
print(node_positions)

['area0', 'area1', 'area2', 'area3', 'cluster0', 'cluster1', 'cluster2', 'cluster3', 'home_landusemix_0', 'home_road_density_0', 'home_openratio_wide_0', 'home_bus_0', 'work_landusemix_0', 'work_road_density_0', 'work_subway_0', 'work_bus_0', 'commute_distance_0', 'home_density_1', 'home_landusemix_1', 'home_road_density_1', 'home_openratio_wide_1', 'home_bus_1', 'work_density_1', 'commute_distance_1', 'gender', 'age', 'edu', 'married', 'income', 'state', 'avg_daily_dur0', 'avg_daily_dur1', 'avg_daily_dur2', 'avg_daily_dur3', 'avg_daily_slow_freq0', 'avg_daily_slow_freq1', 'avg_daily_slow_freq2', 'avg_daily_slow_freq3', 'avg_daily_tube_freq0', 'avg_daily_tube_freq1', 'avg_daily_tube_freq2', 'avg_daily_tube_freq3']


### 设置按类别和阶段分层显示

In [5]:
# 定义分类层级（y轴坐标）
category_levels = {
    'space': 1000,  # 最高层
    'built': 800,
    'user': 600,
    'travel': 400   # 最底层
}

In [27]:
stage_width = 400  # 每个阶段占据的横向宽度
vertical_gap = 300  # 层级间垂直间距

node_attributes = {}
stage_groups = {}

for var in node_positions:
    # 改进阶段提取方式（适配带下划线的变量名）
    stage = 0
    if var[-1].isdigit():
        stage = int(var[-1])
    
    if stage not in stage_groups:
        stage_groups[stage] = []
    stage_groups[stage].append(var)
    
for category in ['space', 'built', 'user', 'travel']:
    # 按阶段排序处理
    for stage in sorted(stage_groups.keys()):
        # 筛选当前分类当前阶段的变量
        vars_in_stage = [v for v in stage_groups[stage] 
                        if (v.startswith('area') or v.startswith('cluster')) and category=='space' or
                           ('home_' in v or 'work_' in v or 'commute' in v) and category=='built' or
                           v in ['gender','age','edu','married','income','state'] and category=='user' or
                           ('avg_daily' in v) and category=='travel']
        
        if not vars_in_stage:
            continue
        
        # 计算横向均匀分布坐标
        num_nodes = len(vars_in_stage)
        x_start = 200 + stage * stage_width
        x_positions = [x_start + (i+1)*stage_width/(num_nodes+1)
                      for i in range(num_nodes)]
        
        # 按字母排序保证布局稳定性
        for var, x in zip(sorted(vars_in_stage), x_positions):
            node_attributes[var] = {
                "x": category_levels[category],
                "y": x,
                "fixed": {"x": False, "y": True},
                "color": {
                    "background": {
                        'space': '#FFB6C1',
                        'built': '#98FB98',
                        'user': '#87CEEB',
                        'travel': '#DDA0DD'
                    }[category]
                }
            }
            

In [87]:
# 配置智能布局参数
layout_config = {
    "physics": {
        "forceAtlas2Based": {
            "gravitationalConstant": -60,  # 负值使节点互相排斥
            "centralGravity": 0.004,
            "springLength": 200,
            "damping": 0.4
        },
        "minVelocity": 0.3,
        "solver": "forceAtlas2Based"
    },
    "configure": {
        "enabled": True,
        "filter": "layout, nodes"  # 允许动态调整布局参数
    }
}

In [88]:
viz2 = plot_structure(
    g, node_attributes=node_attributes,
)
viz2.set_options(options=json.dumps(layout_config))
viz2.show('supporting_files/03_hierarchical.html')

supporting_files/03_hierarchical.html


In [67]:
# 设置白色背景
custom_plot_options = {
    "bgcolor": "white"  # 背景颜色
}

all_edge_attributes = {
    "color":{
        "color": "lightgrey",
    },
    "length": 200,
}

all_node_attributes = {
    "font": {
        "color": "black",
        "face": "Helvetica",
        "size": 30,
        "background": "rgba(211, 211, 211, 0.5)"
    }
}

viz2 = plot_structure(
    g,
    node_attributes=node_attributes,
    all_node_attributes=all_node_attributes,
    all_edge_attributes=all_edge_attributes,
    plot_options=custom_plot_options
)
viz2.set_options(options=json.dumps(layout_config))
viz2.show('supporting_files/03_hierarchical_white.html')

supporting_files/03_hierarchical_white.html


### 修改逻辑

1. 应能清晰区分不同类型和阶段的变量
    * 层级 √
    * 颜色（face & edge & shape）√
    * 大小（中心度）
2. 根据sem结果添加边权重
    * 不显示权重数字（或只显示重要边的权重），而是根据权重设置边样式（系数大小-数字，方向-颜色）
3. 突出显示主要结果

In [23]:
node_attributes

{'area0': {'x': 1000,
  'y': 333.33333333333337,
  'fixed': {'x': False, 'y': True},
  'color': {'background': '#FFB6C1'}},
 'cluster0': {'x': 1000,
  'y': 466.6666666666667,
  'fixed': {'x': False, 'y': True},
  'color': {'background': '#FFB6C1'}},
 'area1': {'x': 1000,
  'y': 733.3333333333334,
  'fixed': {'x': False, 'y': True},
  'color': {'background': '#FFB6C1'}},
 'cluster1': {'x': 1000,
  'y': 866.6666666666667,
  'fixed': {'x': False, 'y': True},
  'color': {'background': '#FFB6C1'}},
 'area2': {'x': 1000,
  'y': 1133.3333333333333,
  'fixed': {'x': False, 'y': True},
  'color': {'background': '#FFB6C1'}},
 'cluster2': {'x': 1000,
  'y': 1266.6666666666667,
  'fixed': {'x': False, 'y': True},
  'color': {'background': '#FFB6C1'}},
 'area3': {'x': 1000,
  'y': 1533.3333333333333,
  'fixed': {'x': False, 'y': True},
  'color': {'background': '#FFB6C1'}},
 'cluster3': {'x': 1000,
  'y': 1666.6666666666667,
  'fixed': {'x': False, 'y': True},
  'color': {'background': '#FFB6C1'}},

#### 修改节点属性

In [31]:
# 个人属性
user_nodes = ['gender', 'age', 'edu', 'married', 'income', 'state']

# 后处理修改user组节点属性
for node_id in node_attributes:
    if node_id in user_nodes:
        # 保留原有坐标等属性，仅覆盖样式
        node_attributes[node_id].update({
            "font": {
                "color": "black",  # RGBA格式（D9表示85%透明度）
                "face": "Helvetica",
                "size": 30,
            },
            "shape": "box",  # 改为方框形状
            "size": 20,      # 节点尺寸
            "borderWidth": 2,
            "color": {
                "border": "#4a90e2",  # 蓝色边框（d9=85%透明度）
                "background": "#FFFFFF", # 深蓝色背景
                # 保留原有highlight设置（如果存在）
                **node_attributes[node_id].get("color", {}).get("highlight", {})
            },
            "mass": 3  # 增加节点质量（物理模拟时不易移动）
        })
        # 合并原有颜色属性（避免覆盖其他颜色设置）
        node_attributes[node_id]["color"] = {
            **node_attributes[node_id].get("color", {}),
            "border": "#4a90e2",
            "background": "#FFFFFF"
        }


In [32]:
viz2 = plot_structure(
    g,
    node_attributes=node_attributes,
    all_node_attributes=all_node_attributes,
    all_edge_attributes=all_edge_attributes,
    plot_options=custom_plot_options
)
viz2.set_options(options=json.dumps(layout_config))
viz2.show('supporting_files/03_hierarchical_modify_user.html')

supporting_files/03_hierarchical_modify_user.html


In [33]:
# space和travel，不同阶段采用不同边界色
# 定义阶段颜色映射（仅space和travel使用）
stage_colors = {
    0: "#80B7DC",  # 蓝
    1: "#F5D98C",  # 黄
    2: "#EFA78D",  # 红色
    3: "#38459B"   # 深蓝
}

def lighten_color(hex_color, factor=0.2):
    """将颜色变浅（factor范围0-1，越大颜色越浅）"""
    # 将HEX转换为归一化的RGB
    rgb = tuple(int(hex_color.lstrip('#')[i:i+2], 16)/255.0 for i in (0, 2, 4))
    # 转换到HLS颜色空间
    h, l, s = colorsys.rgb_to_hls(*rgb)
    # 提升亮度（Lightness）
    new_l = min(0.95, l + factor*(1 - l))
    # 转换回RGB
    new_rgb = colorsys.hls_to_rgb(h, new_l, s)
    
    # 转换回HEX
    return "#{:02x}{:02x}{:02x}".format(
        int(new_rgb[0]*255),
        int(new_rgb[1]*255),
        int(new_rgb[2]*255)
    )

# 仅处理space和travel两类
for node_id in node_attributes:
    # 识别分类
    is_space = node_id.startswith(('area', 'cluster'))
    is_travel = 'avg_daily' in node_id
    
    # 仅处理space和travel
    if not (is_space or is_travel):
        continue
    
    # 提取阶段号（适配两种命名格式）
    stage = 0
    if node_id[-1].isdigit():
        stage = int(node_id[-1])
    
    # 设置边框颜色（保留原有背景色）
    bg_color = "white" if is_space else lighten_color(stage_colors.get(stage, "#4a5568"), factor=0.4) # 根据类型确定颜色
    node_attributes[node_id]['color'] = {
        **node_attributes[node_id].get('color', {}),  # 保留原有颜色
        "background": bg_color,
        "border": stage_colors.get(stage, "#4a5568"),  # 默认灰蓝
        "size": 40,
    }

In [34]:
viz2 = plot_structure(
    g,
    node_attributes=node_attributes,
    all_node_attributes=all_node_attributes,
    all_edge_attributes=all_edge_attributes,
    plot_options=custom_plot_options
)
viz2.set_options(options=json.dumps(layout_config))
viz2.show('supporting_files/03_hierarchical_modify_outcome.html')

supporting_files/03_hierarchical_modify_outcome.html


In [35]:
# 建成环境，使用六边形，不要边界
# 定义built环境变量的阶段颜色（无边框）
built_stage_colors = {
    0: "#15979790",  # 绿
    1: "#F9A02090"   # 黄
}

# 配置built环境变量样式
for node_id in node_attributes:
    # 仅处理建成环境变量（home_/work_/commute开头）
    if not any(k in node_id for k in ('home_', 'work_', 'commute')):
        continue
    
    # 提取阶段号（适配两种命名格式）
    stage = 0
    if node_id[-1].isdigit():
        stage = int(node_id[-1])
    
    # 更新节点属性
    node_attributes[node_id].update({
        "shape": "hexagon",  # 六边形
        "color": {
            "background": built_stage_colors.get(stage, "#98FB98"),  # 默认浅绿
            "border": "rgba(0,0,0,0)", # 完全透明边框
            "highlight": {
                "background": built_stage_colors.get(stage, "#3CB371"),  # 高亮加深
                "border": "rgba(0,0,0,0)"
            }
        },
        "size": 30  # 增大尺寸适应形状
    })

In [36]:
viz2 = plot_structure(
    g,
    node_attributes=node_attributes,
    all_node_attributes=all_node_attributes,
    all_edge_attributes=all_edge_attributes,
    plot_options=custom_plot_options
)
viz2.set_options(options=json.dumps(layout_config))
viz2.show('supporting_files/03_hierarchical_modify_treatment.html')

supporting_files/03_hierarchical_modify_treatment.html


In [80]:
g.degree()

DiDegreeView({'area0': 4, 'area1': 4, 'area2': 3, 'area3': 9, 'cluster0': 7, 'cluster1': 6, 'cluster2': 10, 'cluster3': 12, 'home_landusemix_0': 3, 'home_road_density_0': 5, 'home_openratio_wide_0': 2, 'home_bus_0': 1, 'work_landusemix_0': 2, 'work_road_density_0': 2, 'work_subway_0': 3, 'work_bus_0': 2, 'commute_distance_0': 6, 'home_density_1': 2, 'home_landusemix_1': 2, 'home_road_density_1': 2, 'home_openratio_wide_1': 2, 'home_bus_1': 1, 'work_density_1': 2, 'commute_distance_1': 7, 'gender': 10, 'age': 10, 'edu': 2, 'married': 1, 'income': 9, 'state': 6, 'avg_daily_dur0': 3, 'avg_daily_dur1': 2, 'avg_daily_dur2': 3, 'avg_daily_dur3': 6, 'avg_daily_slow_freq0': 1, 'avg_daily_slow_freq1': 1, 'avg_daily_slow_freq2': 3, 'avg_daily_slow_freq3': 4, 'avg_daily_tube_freq0': 3, 'avg_daily_tube_freq1': 5, 'avg_daily_tube_freq2': 7, 'avg_daily_tube_freq3': 5})

In [37]:
# 根据中心度确定节点大小
# 配置参数
MIN_SIZE = 30  # 最小节点尺寸
MAX_SIZE = 50  # 最大节点尺寸

# 识别目标节点
target_nodes = [
    node for node in node_attributes 
    if node.startswith(('area', 'cluster')) or 'avg_daily' in node
]

# 计算度中心性 (仅针对目标节点)
degree_dict = dict(g.degree())

# 计算尺寸范围
if degree_dict:
    max_degree = max(degree_dict.values())
    min_degree = min(degree_dict.values())
else:
    max_degree = min_degree = 1  # 防止空子图导致的除零错误

# 动态调整目标节点尺寸
for node in target_nodes:
    degree = degree_dict.get(node, 0)
    
    # 线性归一化计算
    if max_degree == min_degree:
        size = (MIN_SIZE + MAX_SIZE) // 2
    else:
        normalized = (degree - min_degree) / (max_degree - min_degree)
        size = MIN_SIZE + normalized * (MAX_SIZE - MIN_SIZE)
    
    # 保留原有形状等其他属性
    node_attributes[node].update({
        "size": int(size),
        # 可选：添加尺寸相关标签缩放
        "font": {
            **node_attributes[node].get("font", {}),
            "size": 40  # 字体随节点尺寸缩放
        }
    })

In [38]:
viz2 = plot_structure(
    g,
    node_attributes=node_attributes,
    all_node_attributes=all_node_attributes,
    all_edge_attributes=all_edge_attributes,
    plot_options=custom_plot_options
)
viz2.set_options(options=json.dumps(layout_config))
viz2.show('supporting_files/03_hierarchical_modify_size.html')

supporting_files/03_hierarchical_modify_size.html


In [39]:
# 精简label

# 处理travel类节点标签
for node_id in node_attributes:
    if 'avg_daily' in node_id:  # 识别travel类节点
        # 去除前缀并更新label
        new_label = node_id.replace("avg_daily_", "", 1)  # 仅替换第一个匹配项
        
        # 保留原其他属性（如颜色、位置等）
        node_attributes[node_id].update({
            "label": new_label,
        })

# 处理be类节点标签
# 定义转换规则
ABBREVIATIONS = {
    'home_': 'H_', 'work_': 'W_',
    'landusemix': 'LUM', 'road_density': 'RD',
    'openratio_wide': 'OR', 'bus': 'BUS',
    'subway': 'SUB', 'density': 'Den',
    'commute_distance': 'CD'
}
PHASE_MAP = {'_0': '(0)', '_1': '(1)'}  # Unicode下标

for var in node_attributes:
    
    if 'home_' in var or 'work_' in var:
        # 阶段处理
        phase = '_' + var.split('_')[-1] if var.split('_')[-1].isdigit() else ''
        phase_symb = PHASE_MAP.get(phase, '')
        
        # 分解指标
        parts = var.replace(phase, '').split('_')
        location = parts[0] + '_'
        metric = '_'.join(parts[1:])
        
        # 生成精简标签
        label = (
            ABBREVIATIONS.get(location, '') + 
            ABBREVIATIONS.get(metric, metric) + 
            phase_symb
        )
        
        # 更新节点属性
        node_attributes[var]['label'] = label
        
    elif 'commute_distance' in var:
        phase = '_' + var.split('_')[-1] if var.split('_')[-1].isdigit() else ''
        phase_symb = PHASE_MAP.get(phase, '')
        label = ABBREVIATIONS['commute_distance'] + phase_symb
        # 更新节点属性
        node_attributes[var]['label'] = label
    
    else:
        continue


In [40]:
viz2 = plot_structure(
    g,
    node_attributes=node_attributes,
    all_node_attributes=all_node_attributes,
    all_edge_attributes=all_edge_attributes,
    plot_options=custom_plot_options
)
viz2.set_options(options=json.dumps(layout_config))
viz2.show('supporting_files/03_hierarchical_modify_label.html')

supporting_files/03_hierarchical_modify_label.html


#### 修改边权重

In [20]:
# 导入结构方程结果
res = pd.read_csv("results/sem_res1.csv")

# 保留回归关系操作符
res = res[res['op']=='~']

res

,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,area1,~,area0,0.429883,0.003710,115.860803,0.0
1,area1,~,cluster1,0.282577,0.003928,71.932650,0.0
2,area2,~,area0,0.088833,0.003584,24.788108,0.0
3,area2,~,area1,0.404127,0.002324,173.889426,0.0
4,area0,~,cluster0,0.263041,0.002933,89.686648,0.0
...,...,...,...,...,...,...,...
85,home_landusemix_1,~,home_openratio_wide_1,-0.231197,0.003157,-73.233766,0.0
86,home_landusemix_1,~,home_bus_1,0.271626,0.003311,82.049685,0.0
87,avg_daily_dur1,~,gender,-0.053395,0.004935,-10.819655,0.0
88,avg_daily_dur1,~,avg_daily_dur0,0.837002,0.002472,338.587793,0.0


In [92]:
# 配置线宽范围
# MIN_WIDTH = 1.0
# MAX_WIDTH = 10.0

# 创建边属性字典
edge_attributes = {}

# 归一化回归系数
# min_estimate = res.Estimate.min()
# max_estimate = res.Estimate.max()

# 生成边粗细映射
for _, row in res.iterrows():
    # 构建边标识（父节点->子节点）
    edge_key = (row['rval'], row['lval'])  # (源节点, 目标节点)
    
    # 归一化计算
    # width = np.interp(row['Estimate'], 
    #                  [min_estimate, max_estimate], 
    #                  [MIN_WIDTH, MAX_WIDTH])
    
    # 设置边属性
    # sig_symbol = '*' * sum([row['p-value'] < alpha for alpha in [0.05, 0.01, 0.001]])
    edge_attributes[edge_key] = {
        'color': {
            'color': 'lightgrey',  # 保持原有灰色
        },
        # 'width': round(width, 2),
        'value': row['Estimate'],
        'label': f"{row['Estimate']:.3f}",  # 显示系数值
        # 'smooth': {'type': 'cubicBezier', 'roundness':0.25}  # 保持曲线平滑
    }
    

In [93]:
# 更新可视化
viz2 = plot_structure(
    g,
    node_attributes=node_attributes,
    edge_attributes=edge_attributes,
    all_node_attributes=all_node_attributes,
    all_edge_attributes=all_edge_attributes,
    plot_options=custom_plot_options
)
viz2.set_options(options=json.dumps(layout_config))
viz2.show('supporting_files/03_hierarchical_modify_edges.html')

supporting_files/03_hierarchical_modify_edges.html


In [94]:
# 微调area的边
area_edges = [
    ("area0", "area1"),
    ("area0", "area3"),
    ("area1", "area3")
]

# 更新边属性（保留原有颜色等设置）
for edge in special_edges:
    # 获取或初始化该边的属性
    edge_attr = edge_attributes.get(edge, {})
    
    # 更新平滑参数
    edge_attr["smooth"] = {
        "enabled": True,
        "type": "curvedCW",  # 顺时针弯曲
        "roundness": 0.15,    # 曲率系数（0-1）
        "forceDirection": "vertical"  # 增强垂直方向弯曲效果
    }
    
    # 保留并合并原有属性
    edge_attributes[edge] = {
        **edge_attr,  # 新设置优先
        **edge_attributes.get(edge, {})  # 保留未修改的原有属性
    }
    
cluster_edges = [
    ("cluster0", "cluster1"),
    ("cluster0", "cluster3"),
    ("cluster1", "cluster3")
]

# 更新边属性（保留原有颜色等设置）
for edge in cluster_edges:
    # 获取或初始化该边的属性
    edge_attr = edge_attributes.get(edge, {})
    
    # 更新平滑参数
    edge_attr["smooth"] = {
        "enabled": True,
        "type": "curvedCCW",  # 顺时针弯曲
        "roundness": 0.15,    # 曲率系数（0-1）
        "forceDirection": "vertical"  # 增强垂直方向弯曲效果
    }
    
    # 保留并合并原有属性
    edge_attributes[edge] = {
        **edge_attr,  # 新设置优先
        **edge_attributes.get(edge, {})  # 保留未修改的原有属性
    }

In [95]:
# 更新可视化
viz2 = plot_structure(
    g,
    node_attributes=node_attributes,
    edge_attributes=edge_attributes,
    all_node_attributes=all_node_attributes,
    all_edge_attributes=all_edge_attributes,
    plot_options=custom_plot_options
)
viz2.set_options(options=json.dumps(layout_config))
viz2.show('supporting_files/03_hierarchical_modify_edges.html')

supporting_files/03_hierarchical_modify_edges.html
